In [0]:
#Importamos las librerias necesarias para trabjar
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
#Parámetro para el entorno dev o prod
dbutils.widgets.text("ENV", "dev")
ENV = dbutils.widgets.get("ENV").strip().lower()

if ENV not in ("dev", "prod"):
    raise ValueError("ENV debe ser 'dev' o 'prod'")

In [0]:
#Configuración de entorno para trabajar
if ENV == "dev":
    CATALOG = "dev-northwind"
    ADLS_ACCOUNT = "adsldevnorthwind"
else:
    CATALOG = "prod-northwind"
    ADLS_ACCOUNT = "adslprodnorthwind"

print("ENV:", ENV)
print("CATALOG:", CATALOG)
print("ADLS_ACCOUNT:", ADLS_ACCOUNT)

In [0]:
#Usamos la base de datos y el schema para poder trabajar
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(F"USE SCHEMA bronze_schema_north")

In [0]:
#Creamos DataFreames para cada tabla para poder trabajarlos de manera independiente a partir de los csv
df_customers = spark.read.format("csv")\
                        .option("header", True)\
                        .option("inferSchema", True)\
                        .load(f"abfss://bronze@{ADLS_ACCOUNT}.dfs.core.windows.net/dbo/Customers/Customers.csv")

df_employees = spark.read.format("csv")\
                        .option("header", True)\
                        .option("inferSchema", True)\
                        .load(f"abfss://bronze@{ADLS_ACCOUNT}.dfs.core.windows.net/dbo/Employees/Employees.csv")

df_order_details = spark.read.format("csv")\
                        .option("header", True)\
                        .option("inferSchema", True)\
                        .load(f"abfss://bronze@{ADLS_ACCOUNT}.dfs.core.windows.net/dbo/Order Details/Order Details.csv")

df_orders = spark.read.format("csv")\
                        .option("header", True)\
                        .option("inferSchema", True)\
                        .load(f"abfss://bronze@{ADLS_ACCOUNT}.dfs.core.windows.net/dbo/Orders/Orders.csv")

df_products = spark.read.format("csv")\
                        .option("header", True)\
                        .option("inferSchema", True)\
                        .load(f"abfss://bronze@{ADLS_ACCOUNT}.dfs.core.windows.net/dbo/Products/Products.csv")


In [0]:
#Creamos una función para escribir los DataFrames en la base de datos y schema correspondiente para la capa bronze
def escribir_to_bronze_table(df, table_name):
  df.write.format("delta")\
  .mode("overwrite")\
  .option("overwriteSchema", "true")\
  .saveAsTable(f"`{CATALOG}`.bronze_schema_north.{table_name}")

#Ejecutamos la función para cada DataFrame 
escribir_to_bronze_table(df_customers, "Customers")
escribir_to_bronze_table(df_employees, "Employees")
escribir_to_bronze_table(df_order_details, "Order_Details")
escribir_to_bronze_table(df_orders, "Orders")
escribir_to_bronze_table(df_products, "Products")


In [0]:
#Cambiamos el schema
spark.sql("USE SCHEMA silver_schema_north")

In [0]:
#Creamos DataFrames para cada tabla para poder trabajarlos de manera independiente a partir de los csv y ejecutar las queries
df_customers              = spark.table(f"`{CATALOG}`.bronze_schema_north.Customers")
df_employees              = spark.table(f"`{CATALOG}`.bronze_schema_north.Employees")
df_order_details          = spark.table(f"`{CATALOG}`.bronze_schema_north.Order_Details")
df_orders                 = spark.table(f"`{CATALOG}`.bronze_schema_north.Orders")
df_products               = spark.table(f"`{CATALOG}`.bronze_schema_north.Products")

**Limpieza Customers**

In [0]:
%sql
USE CATALOG `dev-northwind`;
USE SCHEMA bronze_schema_north;

SELECT * FROM Customers;



In [0]:

#Eliminamos columnas que no vamos a usar
df_customers = df_customers.drop("Fax", "PostalCode")

#organizamos las columnas y renombramos las columnas
df_customers = df_customers.withColumnRenamed("CustomerID", "customer_id")\
                            .withColumnRenamed("CompanyName", "company_name")\
                            .withColumnRenamed("ContactName", "contact_name")\
                            .withColumnRenamed("ContactTitle", "contact_title")\
                            .withColumnRenamed("Address", "address")\
                            .withColumnRenamed("City", "city")\
                            .withColumnRenamed("Region", "region")\
                            .withColumnRenamed("Country", "country")\
                            .withColumnRenamed("Phone", "phone")

#Trim de columnas de texto
df_customers = df_customers\
                .withColumn("company_name", F.trim(F.col("company_name")))\
                .withColumn("contact_name", F.trim(F.col("contact_name")))\
                .withColumn("contact_title", F.trim(F.col("contact_title")))\
                .withColumn("address", F.trim(F.col("address")))\
                .withColumn("city", F.trim(F.col("city")))\
                .withColumn("region", F.trim(F.col("region")))\
                .withColumn("country", F.trim(F.col("country")))

#Eliminamos duplicados
df_customers = df_customers.dropDuplicates(["customer_id"])

#Eliminamos valores nulos
df_customers = df_customers.na.drop()

**Limpieza Employees**

In [0]:
%sql
USE CATALOG `dev-northwind`;
USE SCHEMA bronze_schema_north;

SELECT * FROM employees;

In [0]:
#Eliminamos las columnas que no vamos a utilizar
df_employees = df_employees.drop("address", "city", "region", "country", "homePhone", "extension", "photo", "notes", "photoPath", "ReportsTo")

#organizamos las columnas y renombramos las columnas
df_employees = df_employees.withColumnRenamed("EmployeeID", "employee_id")
df_employees = df_employees.withColumnRenamed("LastName", "last_name")\
                            .withColumnRenamed("FirstName", "first_name")\
                            .withColumnRenamed("Title", "title")\
                            .withColumnRenamed("TitleOfCourtesy", "title_of_courtesy")\
                            .withColumnRenamed("BirthDate", "birth_date")\
                            .withColumnRenamed("HireDate", "hire_date")

#Limpiar Fechas
df_employees = df_employees \
    .withColumn("birth_date", to_date(col("birth_date"))) \
    .withColumn("hire_date",  to_date(col("hire_date")))

#Trim de columnas de texto
df_employees = df_employees\
                .withColumn("last_name", F.trim(F.col("last_name")))\
                .withColumn("first_name", F.trim(F.col("first_name")))\
                .withColumn("title", F.trim(F.col("title")))\
                .withColumn("title_of_courtesy", F.trim(F.col("title_of_courtesy")))

#Eliminamos duplicados
df_employees = df_employees.dropDuplicates()
#Eliminamos valores nulos
df_employees = df_employees.na.drop()


**order_details**

In [0]:
%sql
USE CATALOG `dev-northwind`;
USE SCHEMA bronze_schema_north;

SELECT * FROM order_details;

In [0]:
#Creamos una colummna con el total
df_order_details = df_order_details.withColumn(
    "Total",
    round(
        col("UnitPrice") * col("Quantity") * (1 - col("Discount")),
        2
    )
)

#Eliminamos filas donde Quality sea 0
df_order_details = df_order_details.filter(col("Quantity") > 0)


**Orders**

In [0]:
%sql
USE CATALOG `dev-northwind`;
USE SCHEMA bronze_schema_north;

SELECT * FROM orders;

--df_orders

In [0]:
#organizamos las columnas y renombramos las columnas
df_orders = df_orders.withColumnRenamed("OrderID", "order_id")\
                    .withColumnRenamed("CustomerID", "customer_id")\
                    .withColumnRenamed("EmployeeID", "employee_id")\
                    .withColumnRenamed("OrderDate", "order_date")\
                    .withColumnRenamed("RequiredDate", "required_date")\
                    .withColumnRenamed("ShippedDate", "shipped_date")\
                    .withColumnRenamed("ShipVia", "ship_via")\
                    .withColumnRenamed("Freight", "freight")\
                    .withColumnRenamed("ShipName", "ship_name")\
                    .withColumnRenamed("ShipAddress", "ship_address")\
                    .withColumnRenamed("ShipCity", "ship_city")\
                    .withColumnRenamed("ShipRegion", "ship_region")\
                    .withColumnRenamed("ShipPostalCode", "ship_postal_code")\
                    .withColumnRenamed("ShipCountry", "ship_country")
#Trim de columnas de texto
df_orders = df_orders\
                .withColumn("ship_name", F.trim(F.col("ship_name")))\
                .withColumn("ship_address", F.trim(F.col("ship_address")))\
                .withColumn("ship_city", F.trim(F.col("ship_city")))\
                .withColumn("ship_region", F.trim(F.col("ship_region")))\
                .withColumn("ship_country", F.trim(F.col("ship_country")))

#Limpiar Fechas
df_orders = df_orders \
    .withColumn("order_date", to_date(col("order_date"))) \
    .withColumn("required_date", to_date(col("required_date"))) \
    .withColumn("shipped_date", to_date(col("shipped_date")))

#Eliminamos duplicados
df_orders = df_orders.dropDuplicates(["order_id"])

#Eliminamos valores nulos
df_orders = df_orders.na.drop()

**df_products**

In [0]:
%sql
USE CATALOG `dev-northwind`;
USE SCHEMA bronze_schema_north;

SELECT * FROM products;

--df_products



In [0]:
#organizamos las columnas y renombramos las columnas
df_products = df_products.withColumnRenamed("ProductID", "product_id")\
                        .withColumnRenamed("ProductName", "product_name")\
                        .withColumnRenamed("SupplierID", "supplier_id")\
                        .withColumnRenamed("CategoryID", "category_id")\
                        .withColumnRenamed("QuantityPerUnit", "quantity_per_unit")\
                        .withColumnRenamed("UnitPrice", "unit_price")\
                        .withColumnRenamed("UnitsInStock", "units_in_stock")\
                        .withColumnRenamed("UnitsOnOrder", "units_on_order")\
                        .withColumnRenamed("ReorderLevel", "reorder_level")\
                        .withColumnRenamed("Discontinued", "discontinued")
#Trim de columnas de texto
df_products = df_products\
                .withColumn("product_name", F.trim(F.col("product_name")))\
                .withColumn("quantity_per_unit", F.trim(F.col("quantity_per_unit")))

#Eliminamos duplicados
df_products = df_products.dropDuplicates(["product_id"])
#Eliminamos valores nulos
df_products = df_products.na.drop()

In [0]:
#Usamos la base de datos y el schema para poder trabajar
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(F"USE SCHEMA silver_schema_north")

**Escribir en Silver**

In [0]:
def escribir_to_silver_table(df, table_name):
  df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"`{CATALOG}`.silver_schema_north.{table_name}")

escribir_to_silver_table(df_customers, "customers")
escribir_to_silver_table(df_employees, "employees")
escribir_to_silver_table(df_order_details, "order_details")
escribir_to_silver_table(df_orders, "orders")
escribir_to_silver_table(df_products, "products")

**
Escribir en bronze ADSL**

In [0]:
silver_base_path = (f"abfss://silver@{ADLS_ACCOUNT}.dfs.core.windows.net")

df_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_base_path}/customers")
df_employees.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_base_path}/employees")
df_order_details.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_base_path}/order_details")
df_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_base_path}/orders")
df_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_base_path}/products")

